In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import tensorflow as tf

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report

In [2]:
df=pd.read_csv('cleaned_data.csv')
df.head()

,label,Text
0,0,meeting ã¼ rite ill go home lor ã¼ dun feel li...
1,0,ill always even spirit ill get bb soon trying ...
2,0,sorry took long omw
3,0,thk 50 shd ok said plus minus 10 ã¼ leave line...
4,0,dunno juz askin cos got card got 20 4 salon ca...


In [3]:
df['Text']=df['Text'].fillna("").astype(str)
df=df.dropna(subset=['label'])

In [4]:
X=df['Text']
y=df['label'].astype(int)

In [5]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42, stratify=y)

In [9]:
vectorizer=TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

In [10]:
X_train_vec=vectorizer.fit_transform(X_train)
X_test_vec=vectorizer.transform(X_test)

In [11]:
X_train_np=X_train_vec.toarray().astype(np.float32)
X_test_np=X_test_vec.toarray().astype(np.float32)

y_train_np=y_train.to_numpy().astype(np.float32)
y_test_np=y_test.to_numpy().astype(np.float32)

In [13]:
tf.keras.utils.set_random_seed(42)

In [15]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_np.shape[1],)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

In [16]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=['accuracy']
)

In [17]:
history=model.fit(
    X_train_np,
    y_train_np,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 6s 50ms/step - accuracy: 0.6569 - loss: 0.6690 - val_accuracy: 0.9121 - val_loss: 0.6041
Epoch 2/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.9738 - loss: 0.4126 - val_accuracy: 0.9540 - val_loss: 0.2662
Epoch 3/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.9874 - loss: 0.1050 - val_accuracy: 0.9540 - val_loss: 0.1607
Epoch 4/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9958 - loss: 0.0291 - val_accuracy: 0.9540 - val_loss: 0.1577
Epoch 5/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.9979 - loss: 0.0139 - val_accuracy: 0.9498 - val_loss: 0.1657
Epoch 6/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9990 - loss: 0.0094 - val_accuracy: 0.9456 - val_loss: 0.1735
Epoch 7/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - accuracy: 0.9990 - loss: 0.0074 - val_accuracy: 0.9456 - val_loss: 0.1796
Epoch 8/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.9990 - loss: 0.0063 - val_accuracy: 0.9456 - v

In [20]:
y_prob=model.predict(X_test_np,verbose=0).ravel()

In [21]:
def evaluate_model(y_true, probabilities, threshold=0.5):
    predictions = (np.asarray(probabilities).ravel() >= threshold).astype(int)

    print("Accuracy :", accuracy_score(y_true, predictions))
    print("Precision:", precision_score(y_true, predictions, zero_division=0))
    print("Recall   :", recall_score(y_true, predictions, zero_division=0))
    print("F1-score :", f1_score(y_true, predictions, zero_division=0))
    print("\nClassification Report:\n")
    print(classification_report(y_true, predictions, zero_division=0))


In [23]:
evaluate_model(y_test_np, y_prob)

Accuracy : 0.9498327759197325
Precision: 0.958904109589041
Recall   : 0.9395973154362416
F1-score : 0.9491525423728814

Classification Report:

              precision    recall  f1-score   support

         0.0       0.94      0.96      0.95       150
         1.0       0.96      0.94      0.95       149

    accuracy                           0.95       299
   macro avg       0.95      0.95      0.95       299
weighted avg       0.95      0.95      0.95       299



In [24]:
message = ["Congratulations! You have won a free prize"]
message_vec = vectorizer.transform(message).toarray().astype(np.float32)

probability = model.predict(message_vec, verbose=0)[0, 0]
print("Spam" if probability >= 0.5 else "Not Spam")
print("Spam probability:", round(float(probability), 4))

Spam
Spam probability: 0.9984
